## Observing the CMB

In [ ]:
import maria
from maria.band import get_band

f090 = get_band("act/pa5/f090")
f150 = get_band("act/pa5/f150")


f090.NET_RJ = 10e-6
f150.NET_RJ = 10e-6

f090.knee = 1e1
f150.knee = 1e1

array = {"field_of_view": 0.7,
         "beam_spacing": 1.5,
         "primary_size": 10,
         "packing": "sunflower",
         "shape": "circle",
         "polarized": True,
         "bands": [f090, f150]}

instrument = maria.get_instrument(array=array)

print(instrument)
instrument.plot()

In [ ]:
import numpy as np
from maria import Plan
from maria.plan import *

plan = Plan.generate(duration=600, 
                     sample_rate=20, 
                     start_time="2026-03-05T12:00:00",
                     scan_center=(45, 45), 
                     scan_pattern="raster",
                     scan_options={"x_throw": 2, "y_throw": 2, "speed": 1e0},
                     frame="az/el", 
                     site="cerro_toco")

plan.plot(frames=["az/el", "ra/dec", "glon/glat"])

In [ ]:
sim = maria.Simulation(
    instrument=instrument,
    plans=[plan],
    site="cerro_toco",
    cmb="generate",
    cmb_kwargs={"nside": 2048},
)

print(sim)

In [ ]:
tods = sim.run()
tods[0].plot()

In [ ]:
from maria.mappers.ml_mapper import *

ml_mapper = MaximumLikelihoodMapper(tods=tods,
                                    frame="ra/dec",
                                    resolution=2/60, # degrees
                                    tod_preprocessing={
                                        "remove_spline": {"knot_spacing": 60, "remove_el_gradient_order": 3},
                                    },
                                    bilinear=False,
                                    k=0)

In [ ]:
print(ml_mapper.map)
ml_mapper.map.plot(slices=dict(stokes=["I", "Q", "U"], nu=[[0], [1]]), 
                   cmap="cmb", contrast=1e-4)